In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import tensorflow as tf

# ── Load saved artifacts ──────────────────────────────────────────────────────
artifact_dir = Path('artifacts')

model        = tf.keras.models.load_model(artifact_dir / 'hybrid_model.keras')
preprocessor = joblib.load(artifact_dir / 'hybrid_preprocessor.joblib')
label_enc    = joblib.load(artifact_dir / 'label_encoder.joblib')

print('✅ Artifacts loaded successfully.')
print(f'   Grade classes: {label_enc.classes_.tolist()}')

✅ Artifacts loaded successfully.
   Grade classes: ['A', 'B', 'C', 'D', 'Fail']


In [2]:
FEATURES = [
    (
        'Age',
        'Age',
        '21',
        int,
        None
    ),
    (
        'Gender',
        'Gender',
        'Male',
        str,
        ['Male', 'Female']
    ),
    (
        'Hours_Studied',
        'Hours studied per day (decimal allowed)',
        '5.5',
        float,
        None
    ),
    (
        'Attendance',
        'Attendance percentage (0–100)',
        '80.5',
        float,
        None
    ),
    (
        'Sleep_Hours',
        'Average sleep hours per night (decimal allowed)',
        '7.0',
        float,
        None
    ),
    (
        'Stress_Level',
        'Stress level (1 = low … 10 = high)',
        '4.2',
        float,
        None
    ),
    (
        'Screen_Time',
        'Daily screen time in hours (outside studying)',
        '3.0',
        float,
        None
    ),
    (
        'Previous_GPA',
        'Previous GPA (0.0 – 4.0)',
        '3.1',
        float,
        None
    ),
    (
        'Part_Time_Job',
        'Does the student have a part-time job?',
        'No',
        str,
        ['Yes', 'No']
    ),
    (
        'Study_Method',
        'Preferred study method',
        'Hybrid',
        str,
        ['Online', 'Offline', 'Hybrid']
    ),
    (
        'Diet_Quality',
        'Diet quality',
        'Average',
        str,
        ['Poor', 'Average', 'Good']
    ),
    (
        'Internet_Quality',
        'Internet connection quality',
        'Good',
        str,
        ['Poor', 'Average', 'Good', 'Excellent']
    ),
    (
        'Extracurricular',
        'Participates in extracurricular activities?',
        'Yes',
        str,
        ['Yes', 'No']
    ),
    (
        'Tutoring_Sessions_Per_Week',
        'Number of tutoring sessions per week (integer)',
        '2',
        int,
        None
    ),
    (
        'Family_Income_Level',
        'Family income level',
        'Middle',
        str,
        ['Low', 'Middle', 'High']
    ),
    (
        'Exam_Anxiety_Score',
        'Exam anxiety score (1 = none … 10 = extreme)',
        '3.5',
        float,
        None
    ),
]

def ask_field(label, example, dtype, valid_values):
    if valid_values:
        options = ' / '.join(valid_values)
        prompt  = f'  {label} [{options}]  (e.g. {example}): '
    else:
        prompt  = f'  {label}  (e.g. {example}): '

    while True:
        raw = input(prompt).strip()
        try:
            value = dtype(raw)
        except ValueError:
            print(f'Could not convert "{raw}" to {dtype.__name__}. Try again.')
            continue

        if valid_values and value not in valid_values:
            print(f'"{value}" is not one of {valid_values}. Try again.')
            continue

        return value

print('=' * 60)
print('  STUDENT DATA ENTRY')
print('  Enter each value when prompted. Press Enter to confirm.')
print('=' * 60)
print()

student_data = {}
for col, label, example, dtype, valid in FEATURES:
    student_data[col] = ask_field(label, example, dtype, valid)

student_df = pd.DataFrame([student_data])
print()
print('── Student profile ──────────────────────────────────────')
print(student_df.T.to_string(header=False))

  STUDENT DATA ENTRY
  Enter each value when prompted. Press Enter to confirm.


── Student profile ──────────────────────────────────────
Age                              21
Gender                         Male
Hours_Studied                   4.0
Attendance                     90.0
Sleep_Hours                     6.0
Stress_Level                    4.0
Screen_Time                     3.0
Previous_GPA                    2.0
Part_Time_Job                    No
Study_Method                Offline
Diet_Quality                   Poor
Internet_Quality               Poor
Extracurricular                 Yes
Tutoring_Sessions_Per_Week        2
Family_Income_Level             Low
Exam_Anxiety_Score              2.0


In [3]:
X_student = preprocessor.transform(student_df)

proba   = model.predict(X_student, verbose=0)[0]
pred_idx = np.argmax(proba)
pred_grade = label_enc.classes_[pred_idx]

print('=' * 60)
print('  PREDICTION RESULT')
print('=' * 60)
print(f'  Predicted grade  :  {pred_grade}')
print()
print('  Probability per class:')
for grade, p in zip(label_enc.classes_, proba):
    bar = '█' * int(round(p * 30))
    print(f'    {grade:>4}  {p:.4f}  {bar}')
print('=' * 60)

  PREDICTION RESULT
  Predicted grade  :  B

  Probability per class:
       A  0.3070  █████████
       B  0.6039  ██████████████████
       C  0.0883  ███
       D  0.0006  
    Fail  0.0002  
